In [3]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
import torch

In [4]:
import numpy as np
from tqdm import tqdm
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import accuracy_score, f1_score
from sklearn.preprocessing import StandardScaler


In [5]:
spartun = load_dataset("UKPLab/sparp",'SpaRP-PS1 (SpaRTUN)') # SpaRTUN
stepgame = load_dataset("UKPLab/sparp", "SpaRP-PS2 (StepGame)") # StepGame

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

sparp/SpaRP-PS1 (SpaRTUN)/train.json:   0%|          | 0.00/54.5M [00:00<?, ?B/s]

sparp/SpaRP-PS1 (SpaRTUN)/val.json:   0%|          | 0.00/7.78M [00:00<?, ?B/s]

sparp/SpaRP-PS1 (SpaRTUN)/test.json:   0%|          | 0.00/8.25M [00:00<?, ?B/s]

...

sparp/SpaRP (StepGame)/PS2/val.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

sparp/SpaRP (StepGame)/PS2/test.json:   0%|          | 0.00/313M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/49610 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/4976 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/99299 [00:00<?, ? examples/s]

In [6]:
model_name = "EleutherAI/pythia-2.8b"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(
    model_name,
    output_hidden_states=True,
    torch_dtype=torch.bfloat16
).cuda().eval()

n_layers = model.config.num_hidden_layers  # 32 layers
d_model   = model.config.hidden_size       # 2560
print(f"Layers: {n_layers}, d_model: {d_model}")


config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/396 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/5.68G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/387 [00:00<?, ?it/s]

GPTNeoXModel LOAD REPORT from: EleutherAI/pythia-2.8b
Key              | Status     |  | 
-----------------+------------+--+-
embed_out.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Layers: 32, d_model: 2560


SpaRTRUN Dataset

In [ ]:
print("=== Dataset info ===")
print(spartun)
print()

for i, ex in enumerate(spartun["train"]):
    if i >= 5:
        break
    print(f"--- Example {i} ---")
    for k, v in ex.items():
        print(f"  {k}: {repr(v)}")
    print()

In [ ]:
from collections import Counter

target_counter = Counter()
for ex in list(spartun["train"])[:200]:
    for t in ex["targets"]:
        target_counter[t] += 1

print("Most common target values:")
for val, count in target_counter.most_common(20):
    print(f"  {repr(val):30s}  {count}")

Most common target values:
  'outside'                       55
  'below'                         53
  'above'                         50
  'far'                           36
  'inside'                        27
  'right'                         23
  'behind'                        21
  'in front'                      20
  'contains'                      19
  'left'                          18
  'inside and touching'           10
  'outside and touching'          7
  'contains and touches'          6
  'near'                          5
  'partially overlapping'         1


In [ ]:
ex = spartun["train"][0]
context  = ex["context"]
relation = ex["targets"][0]

print("Context:  ", context[:200])
print("Relation: ", repr(relation))
print()

# What the context tokenizes to
ctx_ids     = tokenizer(context, add_special_tokens=False)["input_ids"]
ctx_tokens  = tokenizer.convert_ids_to_tokens(ctx_ids)
print("Context tokens:", ctx_tokens[:40])
print()

# What the relation tokenizes to (standalone)
span_ids    = tokenizer(relation, add_special_tokens=False)["input_ids"]
span_tokens = tokenizer.convert_ids_to_tokens(span_ids)
print("Relation token ids (standalone):", span_ids)
print("Relation tokens (standalone):   ", span_tokens)
print()

# manual search
found = False
for i in range(len(ctx_ids) - len(span_ids) + 1):
    if ctx_ids[i : i + len(span_ids)] == span_ids:
        print(f"Found at position {i}: {ctx_tokens[i:i+len(span_ids)]}")
        found = True
        break
if not found:
    print("NOT FOUND — checking if any individual span token appears:")
    for tok_id, tok in zip(span_ids, span_tokens):
        positions = [j for j, c in enumerate(ctx_ids) if c == tok_id]
        print(f"  token {repr(tok)} (id={tok_id}) appears at positions: {positions}")



Context:   Two boxes, called one and two exist in the image. Box one has a medium yellow melon. To the south of the medium yellow melon is a small yellow watermelon. The small yellow watermelon is in box one. Bo
Relation:  'inside'

Context tokens: ['Two', 'Ġboxes', ',', 'Ġcalled', 'Ġone', 'Ġand', 'Ġtwo', 'Ġexist', 'Ġin', 'Ġthe', 'Ġimage', '.', 'ĠBox', 'Ġone', 'Ġhas', 'Ġa', 'Ġmedium', 'Ġyellow', 'Ġmel', 'on', '.', 'ĠTo', 'Ġthe', 'Ġsouth', 'Ġof', 'Ġthe', 'Ġmedium', 'Ġyellow', 'Ġmel', 'on', 'Ġis', 'Ġa', 'Ġsmall', 'Ġyellow', 'Ġwaterm', 'elon', '.', 'ĠThe', 'Ġsmall', 'Ġyellow']

Relation token ids (standalone): [40084]
Relation tokens (standalone):    ['inside']

NOT FOUND — checking if any individual span token appears:
  token 'inside' (id=40084) appears at positions: []


In [ ]:
ex = spartun["train"][0]
for k, v in ex.items():
    print(f"{k}: {repr(v)[:200]}")

commonsense_question: ''
spatial_types: []
canary: ''
targets: ['inside']
num_question_entities: 2
num_hop: 2
target_scores: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0]
reasoning: 'Step 1: From the context, the box one contains the medium yellow melon.\nStep 2: From step 1, we can say that the medium yellow melon is inside the box one.\nStep 3: It is given that the box two cont
target_choices: ['left', 'right', 'above', 'below', 'behind', 'in front', 'near', 'far', 'outside', 'outside and touching', 'partially overlapping', 'inside and touching', 'inside', 'contains and touches', 'contains'
question_type: 'FR'
context: 'Two boxes, called one and two exist in the image. Box one has a medium yellow melon. To the south of the medium yellow melon is a small yellow watermelon. The small yellow watermelon is in box one. B
reasoning_types: []
symbolic_context: '{"0-->-1": ["ntpp"], "1-->-1": ["ntpp"], "0-->0x0": ["ntppi"], "0x1-->0x0": ["below"], "0x1-->0": ["ntpp"], "1-->1x1": ["ntppi"]

In [ ]:
def extract_hidden_states(text):
    """
    Mean-pool over all non-padding token positions at each layer.
    Much more stable signal than last-token for classification probing.
    """
    inputs = tokenizer(
        text, return_tensors="pt",
        truncation=True, max_length=512
    ).to("cuda")

    attention_mask = inputs["attention_mask"]   # (1, seq_len)

    with torch.no_grad():
        outputs = model(**inputs)

    layer_states = []
    for layer_hidden in outputs.hidden_states[1:]:
        # layer_hidden: (1, seq_len, d_model)
        mask = attention_mask.unsqueeze(-1).float()           # (1, seq_len, 1)
        summed = (layer_hidden * mask).sum(dim=1)             # (1, d_model)
        counts = mask.sum(dim=1).clamp(min=1)                 # (1, 1)
        h = (summed / counts)[0].cpu().float().numpy()        # (d_model,)
        layer_states.append(h)

    return layer_states

In [ ]:
ex = spartun["train"][0]
print("symbolic_question:", ex.get("symbolic_question"))
print("context snippet:  ", ex["context"][:300])
print("targets:          ", ex["targets"])

symbolic_question: ['0x0', '1']
context snippet:   Two boxes, called one and two exist in the image. Box one has a medium yellow melon. To the south of the medium yellow melon is a small yellow watermelon. The small yellow watermelon is in box one. Box two with a medium orange fruit has box one. A small yellow melon is inside this box. This fruit is
targets:           ['inside']


In [ ]:
# RELATION_LABELS = {
#     "left":                    0,
#     "right":                   1,
#     "above":                   2,
#     "below":                   3,
#     "in front":                4,
#     "behind":                  5,
#     "near":                    6,
#     "far":                     7,
#     "inside":                  8,
#     "outside":                 9,
#     "contains":                10,
#     "inside and touching":     11,
#     "outside and touching":    12,
#     "contains and touches":    13,
#     "partially overlapping":   14,
# }

RELATION_LABELS = {
    # Directional — left/right/above/below
    "left":                    0,
    "right":                   0,
    "above":                   1,
    "below":                   1,
    # Depth
    "in front":                2,
    "behind":                  2,
    # Proximity
    "near":                    3,
    "far":                     3,
    # Topological (inside/outside/overlap variants)
    "inside":                  4,
    "outside":                 4,
    "contains":                4,
    "inside and touching":     4,
    "outside and touching":    4,
    "contains and touches":    4,
    "partially overlapping":   4,
}

all_states = [[] for _ in range(n_layers)]
all_labels = []
skipped = 0

for example in tqdm(spartun["train"]):
    targets = example["targets"]
    if len(targets) != 1:
        skipped += 1
        continue
    relation = targets[0]
    if relation not in RELATION_LABELS:
        skipped += 1
        continue

    hs = extract_hidden_states(example["context"])

    for L, h in enumerate(hs):
        all_states[L].append(h)
    all_labels.append(RELATION_LABELS[relation])

y = np.array(all_labels)
X_per_layer = [np.stack(s) for s in all_states]

print(f"Collected {len(y)} examples  |  skipped {skipped}")

label_names = ["lateral (L/R)", "vertical (up/down)", "depth (front/back)",
               "proximity (near/far)", "topological"]
print("\nClass distribution:")
for i, name in enumerate(label_names):
    print(f"  {i} {name:25s} {int((y==i).sum())}")

100%|██████████| 16450/16450 [06:43<00:00, 40.78it/s]


Collected 10866 examples  |  skipped 5584

Class distribution:
  0 lateral (L/R)             1065
  1 vertical (up/down)        3172
  2 depth (front/back)        443
  3 proximity (near/far)      300
  4 topological               5886


In [ ]:
probe_acc = []
probe_f1  = []
probes    = []   #to save the trained model later for inference

sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

for L in range(n_layers):
    X = X_per_layer[L]
    train_idx, test_idx = next(sss.split(X, y))

    scaler  = StandardScaler()
    X_train = scaler.fit_transform(X[train_idx])
    X_test  = scaler.transform(X[test_idx])
    y_train, y_test = y[train_idx], y[test_idx]

    probe = LogisticRegression(max_iter=1000, C=1.0, n_jobs=-1)
    probe.fit(X_train, y_train)
    y_pred = probe.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    f1  = f1_score(y_test, y_pred, average="macro", zero_division=0)
    probe_acc.append(acc)
    probe_f1.append(f1)
    probes.append((scaler, probe))

    print(f"Layer {L+1:02d}  acc={acc:.3f}  macro-F1={f1:.3f}")

peak_layer = int(np.argmax(probe_acc)) + 1
print(f"\nPeak accuracy at layer {peak_layer}: {probe_acc[peak_layer-1]:.3f}")

Layer 01  acc=0.537  macro-F1=0.394
Layer 02  acc=0.506  macro-F1=0.373
Layer 03  acc=0.472  macro-F1=0.342
Layer 04  acc=0.490  macro-F1=0.361
Layer 05  acc=0.488  macro-F1=0.339
Layer 06  acc=0.483  macro-F1=0.341
Layer 07  acc=0.479  macro-F1=0.334
Layer 08  acc=0.483  macro-F1=0.339
Layer 09  acc=0.475  macro-F1=0.335
Layer 10  acc=0.479  macro-F1=0.336
Layer 11  acc=0.457  macro-F1=0.314
Layer 12  acc=0.460  macro-F1=0.326
Layer 13  acc=0.465  macro-F1=0.329
Layer 14  acc=0.460  macro-F1=0.326
Layer 15  acc=0.472  macro-F1=0.334
Layer 16  acc=0.466  macro-F1=0.338
Layer 17  acc=0.473  macro-F1=0.338
Layer 18  acc=0.481  macro-F1=0.340
Layer 19  acc=0.476  macro-F1=0.332
Layer 20  acc=0.474  macro-F1=0.331
Layer 21  acc=0.467  macro-F1=0.336
Layer 22  acc=0.481  macro-F1=0.337
Layer 23  acc=0.470  macro-F1=0.327
Layer 24  acc=0.479  macro-F1=0.331
Layer 25  acc=0.468  macro-F1=0.330
Layer 26  acc=0.473  macro-F1=0.326
Layer 27  acc=0.466  macro-F1=0.326
Layer 28  acc=0.469  macro-F

In [ ]:
from collections import Counter
import numpy as np

counts = Counter(y.tolist())
majority_acc = max(counts.values()) / len(y)
print(f"Majority class baseline: {majority_acc:.3f}")
print(f"Layer 1 accuracy:        {probe_acc[0]:.3f}")
print(f"Layer 32 accuracy:       {probe_acc[31]:.3f}")
print(f"Chance (uniform):        {1/5:.3f}")
print()
print("Class counts:")
label_names = ["lateral (L/R)", "vertical (up/down)", "depth (front/back)",
               "proximity (near/far)", "topological"]
for i, name in enumerate(label_names):
    print(f"  {name:25s}  n={counts[i]}")

Majority class baseline: 0.542
Layer 1 accuracy:        0.537
Layer 32 accuracy:       0.458
Chance (uniform):        0.200

Class counts:
  lateral (L/R)              n=1065
  vertical (up/down)         n=3172
  depth (front/back)         n=443
  proximity (near/far)       n=300
  topological                n=5886


In [ ]:
from sklearn.metrics import classification_report

sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

for L in [0, 15, 31]:   # first, middle, last
    X = X_per_layer[L]
    train_idx, test_idx = next(sss.split(X, y))
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X[train_idx])
    X_test  = scaler.transform(X[test_idx])
    y_train, y_test = y[train_idx], y[test_idx]

    probe = LogisticRegression(max_iter=1000, C=1.0, n_jobs=-1)
    probe.fit(X_train, y_train)
    y_pred = probe.predict(X_test)

    print(f"\n{'='*50}")
    print(f"Layer {L+1}")
    print(classification_report(y_test, y_pred,
          target_names=label_names, zero_division=0))


Layer 1
                      precision    recall  f1-score   support

       lateral (L/R)       0.43      0.45      0.44       213
  vertical (up/down)       0.47      0.39      0.42       635
  depth (front/back)       0.39      0.44      0.41        88
proximity (near/far)       0.06      0.05      0.06        60
         topological       0.61      0.67      0.64      1178

            accuracy                           0.54      2174
           macro avg       0.39      0.40      0.39      2174
        weighted avg       0.53      0.54      0.53      2174


Layer 16
                      precision    recall  f1-score   support

       lateral (L/R)       0.35      0.35      0.35       213
  vertical (up/down)       0.35      0.32      0.34       635
  depth (front/back)       0.35      0.43      0.39        88
proximity (near/far)       0.03      0.03      0.03        60
         topological       0.57      0.59      0.58      1178

            accuracy                          

In [ ]:
import plotly.graph_objects as go

layers  = list(range(1, n_layers + 1))
chance  = 1 / 5

fig = go.Figure()

fig.add_trace(go.Bar(
    x=layers, y=probe_acc,
    name="Accuracy",
    marker_color=[
        f"rgba(83,74,183,{0.3 + 0.7*(a/max(probe_acc))})"
        for a in probe_acc
    ],
))
fig.add_trace(go.Scatter(
    x=layers, y=probe_f1,
    name="Macro F1",
    mode="lines+markers",
    line=dict(color="#1D9E75", width=2),
    marker=dict(size=5),
))
fig.add_hline(y=chance, line_dash="dash", line_color="gray",
              annotation_text="Chance (0.20)", annotation_position="bottom right")
fig.add_hline(y=majority_acc, line_dash="dot", line_color="orange",
              annotation_text=f"Majority baseline ({majority_acc:.2f})",
              annotation_position="top right")
fig.add_annotation(
    x=1, y=probe_acc[0],
    text=f"L1 peak: {probe_acc[0]:.3f}",
    showarrow=True, arrowhead=2, ax=40, ay=-30
)
fig.update_layout(
    title="Spatial probe accuracy — Pythia-2.8B on SpaRTUN<br>"
          "<sup>Early layers encode spatial relations most linearly</sup>",
    xaxis_title="Layer",
    yaxis_title="Score",
    yaxis=dict(range=[0, 0.7]),
    legend=dict(orientation="h", yanchor="bottom", y=1.02),
    height=450,
)
fig.show()

StepGame Dataset

In [7]:
ex = stepgame["train"][0]
for k, v in ex.items():
    print(f"{k}: {repr(str(v)[:150])}")

reasoning: 'Step 1: It is given that X is left of K.'
source_data: 'StepGame'
target_scores: '[0, 0, 0, 0, 1]'
comments: "['point_of_view_type: Fixed Orientation Point of View (FPoV)', 'entity_type: Point Objects (PO)', 'relation_type: Relation Complete (RC)', 'quantitati"
symbolic_context: '{"X-->K": ["left"]}'
symbolic_entity_map: '{"X": "X", "K": "K"}'
symbolic_question: "['X', 'K']"
context_id: 'train/0'
targets: "['left']"
question: 'What is the relation of the agent X to the agent K?'
reasoning_types: '[]'
question_id: '1'
num_question_entities: '2'
target_choices: "['above', 'below', 'right', 'overlapping', 'left']"
symbolic_reasoning: '[{"head": "X", "tail": "K", "context_rel": {"left": {"unit": 1}}, "inv_context_rel": {}, "inferred_rel": {}}]'
canary: ''
commonsense_question: ''
num_context_entities: '2'
question_type: 'FR'
num_hop: '1'
context: 'X is to the left of K and is on the same horizontal plane.'
spatial_types: '[]'


In [8]:
from collections import Counter

target_counter = Counter()
for ex in list(stepgame["train"])[:500]:
    for t in ex["targets"]:
        target_counter[t] += 1

print("StepGame target values:")
for val, count in target_counter.most_common(20):
    print(f"  {repr(val):30s}  {count}")

StepGame target values:
  'left'                          211
  'below'                         194
  'above'                         182
  'right'                         165


In [9]:

STEPGAME_LABELS = {
    "left":   0,
    "right":  1,
    "above":  2,
    "below":  3,
}
sg_label_names = ["left", "right", "above", "below"]

In [ ]:
# all_states_sg = [[] for _ in range(n_layers)]
# all_labels_sg = []
# skipped_sg    = 0

# for example in tqdm(stepgame["train"]):
#     targets = example["targets"]

#     # targets is stored as a string "['left']" — parse it
#     if isinstance(targets, str):
#         import ast
#         targets = ast.literal_eval(targets)

#     if len(targets) != 1:
#         skipped_sg += 1
#         continue
#     relation = targets[0]
#     if relation not in STEPGAME_LABELS:
#         skipped_sg += 1
#         continue

#     hs = extract_hidden_states(example["context"])
#     for L, h in enumerate(hs):
#         all_states_sg[L].append(h)
#     all_labels_sg.append(STEPGAME_LABELS[relation])

# y_sg = np.array(all_labels_sg)
# X_per_layer_sg = [np.stack(s) for s in all_states_sg]

# majority_sg = max(Counter(y_sg.tolist()).values()) / len(y_sg)
# print(f"Collected {len(y_sg)} examples  |  skipped {skipped_sg}")
# print(f"Chance baseline:   {1/4:.3f}")
# print(f"Majority baseline: {majority_sg:.3f}")
# print(f"\nClass distribution:")
# for i, name in enumerate(sg_label_names):
#     print(f"  {name:10s}  n={int((y_sg==i).sum())}")

100%|██████████| 49610/49610 [13:19<00:00, 62.04it/s]


Collected 22664 examples  |  skipped 26946
Chance baseline:   0.250
Majority baseline: 0.256

Class distribution:
  left        n=5696
  right       n=5610
  above       n=5812
  below       n=5546


In [ ]:
# probe_acc_sg = []
# probe_f1_sg  = []
# probes_sg    = []

# sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

# for L in range(n_layers):
#     X = X_per_layer_sg[L]
#     train_idx, test_idx = next(sss.split(X, y_sg))

#     scaler  = StandardScaler()
#     X_train = scaler.fit_transform(X[train_idx])
#     X_test  = scaler.transform(X[test_idx])
#     y_train, y_test = y_sg[train_idx], y_sg[test_idx]

#     probe = LogisticRegression(max_iter=1000, C=1.0, n_jobs=-1)
#     probe.fit(X_train, y_train)
#     y_pred = probe.predict(X_test)

#     acc = accuracy_score(y_test, y_pred)
#     f1  = f1_score(y_test, y_pred, average="macro", zero_division=0)
#     probe_acc_sg.append(acc)
#     probe_f1_sg.append(f1)
#     probes_sg.append((scaler, probe))

#     print(f"Layer {L+1:02d}  acc={acc:.3f}  macro-F1={f1:.3f}")

# peak_sg = int(np.argmax(probe_acc_sg)) + 1
# print(f"\nPeak accuracy at layer {peak_sg}: {probe_acc_sg[peak_sg-1]:.3f}")
# print(f"Majority baseline:              {majority_sg:.3f}")

Layer 01  acc=0.354  macro-F1=0.353
Layer 02  acc=0.341  macro-F1=0.342
Layer 03  acc=0.351  macro-F1=0.351
Layer 04  acc=0.339  macro-F1=0.339
Layer 05  acc=0.342  macro-F1=0.342
Layer 06  acc=0.337  macro-F1=0.337
Layer 07  acc=0.340  macro-F1=0.340
Layer 08  acc=0.335  macro-F1=0.335
Layer 09  acc=0.335  macro-F1=0.334
Layer 10  acc=0.333  macro-F1=0.332
Layer 11  acc=0.346  macro-F1=0.346
Layer 12  acc=0.334  macro-F1=0.334
Layer 13  acc=0.326  macro-F1=0.326
Layer 14  acc=0.340  macro-F1=0.340
Layer 15  acc=0.344  macro-F1=0.344
Layer 16  acc=0.341  macro-F1=0.341
Layer 17  acc=0.340  macro-F1=0.339
Layer 18  acc=0.341  macro-F1=0.340
Layer 19  acc=0.338  macro-F1=0.338
Layer 20  acc=0.344  macro-F1=0.344
Layer 21  acc=0.347  macro-F1=0.347
Layer 22  acc=0.347  macro-F1=0.347
Layer 23  acc=0.354  macro-F1=0.354
Layer 24  acc=0.349  macro-F1=0.349
Layer 25  acc=0.347  macro-F1=0.347
Layer 26  acc=0.343  macro-F1=0.343
Layer 27  acc=0.339  macro-F1=0.339
Layer 28  acc=0.341  macro-F

In [ ]:

# sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

# for L in [0, peak_sg - 1, 31]:
#     X = X_per_layer_sg[L]
#     train_idx, test_idx = next(sss2.split(X, y_sg))
#     scaler = StandardScaler()
#     X_train = scaler.fit_transform(X[train_idx])
#     X_test  = scaler.transform(X[test_idx])

#     probe = LogisticRegression(max_iter=1000, C=1.0, n_jobs=-1)
#     probe.fit(X_train, y_sg[train_idx])
#     y_pred = probe.predict(X_test)

#     print(f"\n{'='*45}")
#     print(f"Layer {L+1}")
#     print(classification_report(
#         y_sg[test_idx], y_pred,
#         target_names=sg_label_names, zero_division=0
#     ))


Layer 1
              precision    recall  f1-score   support

        left       0.36      0.35      0.35      1139
       right       0.35      0.37      0.36      1122
       above       0.36      0.37      0.37      1163
       below       0.34      0.33      0.33      1109

    accuracy                           0.35      4533
   macro avg       0.35      0.35      0.35      4533
weighted avg       0.35      0.35      0.35      4533


Layer 1
              precision    recall  f1-score   support

        left       0.36      0.35      0.35      1139
       right       0.35      0.37      0.36      1122
       above       0.36      0.37      0.37      1163
       below       0.34      0.33      0.33      1109

    accuracy                           0.35      4533
   macro avg       0.35      0.35      0.35      4533
weighted avg       0.35      0.35      0.35      4533


Layer 32
              precision    recall  f1-score   support

        left       0.36      0.37      0.36    

In [10]:
import ast

ex = stepgame["train"][0]
print("context:          ", ex["context"])
print("question:         ", ex["question"])
print("symbolic_question:", ex["symbolic_question"])   # entity pair
print("targets:          ", ex["targets"])
print("num_hop:          ", ex["num_hop"])

context:           X is to the left of K and is on the same horizontal plane.
question:          What is the relation of the agent X to the agent K?
symbolic_question: ['X', 'K']
targets:           ['left']
num_hop:           1


In [11]:
# The question asks "what is the relation of X to K?"
# The model encodes that relation in the hidden state of the object entity (K)
# after reading the full context — that's the most task-relevant position

def get_entity_positions(input_ids, entity_str, tokenizer):
    """Find the LAST occurrence of entity_str tokens in input_ids."""
    span_ids_a = tokenizer(" " + entity_str,
                           add_special_tokens=False)["input_ids"]
    span_ids_b = tokenizer(entity_str,
                           add_special_tokens=False)["input_ids"]

    found = None
    for span_ids in [span_ids_a, span_ids_b]:
        for i in range(len(input_ids) - len(span_ids) + 1):
            if input_ids[i : i + len(span_ids)] == span_ids:
                found = (i, i + len(span_ids))
                # keep searching to get the LAST occurrence
    return found   # (start, end) or None


def extract_hidden_states_entity(context, entity_a, entity_b):
    """
    Extract hidden states mean-pooled over both entity token spans.
    Falls back to last-token if entities not found.
    """
    inputs = tokenizer(context, return_tensors="pt",
                       truncation=True, max_length=512).to("cuda")
    input_ids = inputs["input_ids"][0].tolist()

    pos_a = get_entity_positions(input_ids, entity_a, tokenizer)
    pos_b = get_entity_positions(input_ids, entity_b, tokenizer)

    with torch.no_grad():
        outputs = model(**inputs)

    layer_states = []
    for layer_hidden in outputs.hidden_states[1:]:
        h = layer_hidden[0]   # (seq_len, d_model)

        vectors = []
        if pos_a:
            vectors.append(h[pos_a[0]:pos_a[1]].mean(0))
        if pos_b:
            vectors.append(h[pos_b[0]:pos_b[1]].mean(0))
        if not vectors:
            vectors.append(h[-1])   # fallback: last token

        # Concatenate entity representations — encodes both + their difference
        if len(vectors) == 2:
            combined = torch.cat(vectors, dim=0)   # (2*d_model,)
        else:
            combined = vectors[0]

        layer_states.append(combined.cpu().float().numpy())

    return layer_states

In [12]:
all_states_sg2 = [[] for _ in range(n_layers)]
all_labels_sg2 = []
skipped_sg2    = 0
fallback_count = 0

for example in tqdm(stepgame["train"]):
    targets = example["targets"]
    if isinstance(targets, str):
        targets = ast.literal_eval(targets)
    if len(targets) != 1:
        skipped_sg2 += 1
        continue
    relation = targets[0]
    if relation not in STEPGAME_LABELS:
        skipped_sg2 += 1
        continue

    # Parse the entity pair from symbolic_question: "['X', 'K']"
    sq = example["symbolic_question"]
    if isinstance(sq, str):
        sq = ast.literal_eval(sq)
    if len(sq) < 2:
        skipped_sg2 += 1
        continue
    entity_a, entity_b = sq[0], sq[1]

    hs = extract_hidden_states_entity(example["context"], entity_a, entity_b)

    for L, h in enumerate(hs):
        all_states_sg2[L].append(h)
    all_labels_sg2.append(STEPGAME_LABELS[relation])

y_sg2 = np.array(all_labels_sg2)
X_per_layer_sg2 = [np.stack(s) for s in all_states_sg2]

print(f"Collected {len(y_sg2)} examples  |  skipped {skipped_sg2}")
print(f"Chance: {1/4:.3f}")

 70%|███████   | 34948/49610 [09:25<03:57, 61.77it/s]

KeyboardInterrupt



In [ ]:
probe_acc_sg2 = []
probe_f1_sg2  = []
probes_sg2    = []

sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

for L in range(n_layers):
    X = X_per_layer_sg2[L]
    train_idx, test_idx = next(sss.split(X, y_sg2))  # split against y_sg2

    scaler  = StandardScaler()
    X_train = scaler.fit_transform(X[train_idx])
    X_test  = scaler.transform(X[test_idx])
    y_train = y_sg2[train_idx]   # use y_sg2, not y from previous dataset
    y_test  = y_sg2[test_idx]    # define y_test explicitly here

    probe = LogisticRegression(max_iter=1000, C=1.0, n_jobs=-1)
    probe.fit(X_train, y_train)
    y_pred = probe.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    f1  = f1_score(y_test, y_pred, average="macro", zero_division=0)
    probe_acc_sg2.append(acc)
    probe_f1_sg2.append(f1)
    probes_sg2.append((scaler, probe))

    print(f"Layer {L+1:02d}  acc={acc:.3f}  macro-F1={f1:.3f}")

peak_sg2 = int(np.argmax(probe_acc_sg2)) + 1
print(f"\nPeak at layer {peak_sg2}: {probe_acc_sg2[peak_sg2-1]:.3f}")
print(f"Chance baseline: {1/4:.3f}")

In [ ]:
import plotly.graph_objects as go

layers = list(range(1, n_layers + 1))

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=layers, y=probe_acc_sg2,
    name="Accuracy",
    mode="lines+markers",
    line=dict(color="#D85A30", width=2),
    marker=dict(size=5),
))
fig.add_trace(go.Scatter(
    x=layers, y=probe_f1_sg2,
    name="Macro-F1",
    mode="lines+markers",
    line=dict(color="#534AB7", width=2, dash="dot"),
    marker=dict(size=4),
))

fig.add_hline(y=0.25, line_dash="dash", line_color="gray", opacity=0.5,
              annotation_text="Chance (0.25)",
              annotation_position="bottom right")

fig.add_annotation(
    x=1, y=probe_acc_sg2[0],
    text=f"L1: {probe_acc_sg2[0]:.3f}",
    showarrow=True, arrowhead=2, ax=40, ay=-30,
)
fig.add_annotation(
    x=32, y=probe_acc_sg2[31],
    text=f"L32: {probe_acc_sg2[31]:.3f}",
    showarrow=True, arrowhead=2, ax=-40, ay=-30,
)

fig.update_layout(
    title="Spatial relation probe — StepGame entity-targeted · Pythia-2.8B",
    xaxis_title="Layer",
    yaxis_title="Score",
    yaxis=dict(range=[0, 0.7]),
    legend=dict(orientation="h", yanchor="bottom", y=1.02),
    height=420,
)
fig.show()

In [ ]:
from sklearn.metrics import classification_report
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

for L in [0, 15, 31]:
    X = X_per_layer_sg2[L]
    train_idx, test_idx = next(sss.split(X, y_sg2))

    scaler  = StandardScaler()
    X_train = scaler.fit_transform(X[train_idx])
    X_test  = scaler.transform(X[test_idx])

    probe = LogisticRegression(max_iter=1000, C=1.0, n_jobs=-1)
    probe.fit(X_train, y_sg2[train_idx])
    y_pred = probe.predict(X_test)

    print(f"\n{'='*45}")
    print(f"Layer {L+1}")
    print(classification_report(
        y_sg2[test_idx], y_pred,
        target_names=sg_label_names, zero_division=0
    ))

In [13]:
# Splitting examples by num_hop and train a probe per (hop, layer) combination
# Then we plot how the layer-accuracy curve shifts as k increases

import ast
import numpy as np
from collections import defaultdict
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from tqdm import tqdm
import plotly.graph_objects as go

# We need (hidden_states, relation_label, k_hop) per example

all_states_hop  = [[] for _ in range(n_layers)]
all_labels_hop  = []
all_hops        = []
skipped_hop     = 0

for example in tqdm(stepgame["train"]):
    targets = example["targets"]
    if isinstance(targets, str):
        targets = ast.literal_eval(targets)
    if len(targets) != 1:
        skipped_hop += 1
        continue
    relation = targets[0]
    if relation not in STEPGAME_LABELS:
        skipped_hop += 1
        continue

    sq = example["symbolic_question"]
    if isinstance(sq, str):
        sq = ast.literal_eval(sq)
    if len(sq) < 2:
        skipped_hop += 1
        continue

    k = int(example["num_hop"])
    entity_a, entity_b = sq[0], sq[1]

    hs = extract_hidden_states_entity(example["context"], entity_a, entity_b)

    for L, h in enumerate(hs):
        all_states_hop[L].append(h)
    all_labels_hop.append(STEPGAME_LABELS[relation])
    all_hops.append(k)

y_hop   = np.array(all_labels_hop)
hops    = np.array(all_hops)
X_hop   = [np.stack(s) for s in all_states_hop]

print(f"Collected {len(y_hop)} examples  |  skipped {skipped_hop}")
print(f"\nHop distribution:")
for k in sorted(np.unique(hops)):
    print(f"  k={k}  n={int((hops==k).sum())}")

100%|██████████| 49610/49610 [13:04<00:00, 63.23it/s]


Collected 22664 examples  |  skipped 26946

Hop distribution:
  k=1  n=14347
  k=2  n=5389
  k=3  n=1927
  k=4  n=788
  k=5  n=213


In [14]:
# Only keeps hop values with enough examples for stratified split (min ~50)

MIN_PER_CLASS = 10   # minimum per class per hop group to train reliably
hop_values    = sorted([k for k in np.unique(hops)
                        if (hops==k).sum() >= MIN_PER_CLASS * 4])
print(f"Running hop analysis for k = {hop_values}")

# acc_by_hop[k] = list of accuracies across 32 layers
acc_by_hop = {}

for k in hop_values:
    mask   = hops == k
    y_k    = y_hop[mask]
    accs_k = []

    for L in range(n_layers):
        X_k = X_hop[L][mask]

        # Need enough examples per class for stratified split
        class_counts = [int((y_k==c).sum()) for c in range(4)]
        if min(class_counts) < 5:
            accs_k.append(np.nan)
            continue

        try:
            sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
            train_idx, test_idx = next(sss.split(X_k, y_k))
        except ValueError:
            accs_k.append(np.nan)
            continue

        scaler  = StandardScaler()
        X_train = scaler.fit_transform(X_k[train_idx])
        X_test  = scaler.transform(X_k[test_idx])

        probe = LogisticRegression(max_iter=1000, C=1.0, n_jobs=-1)
        probe.fit(X_train, y_k[train_idx])
        y_pred = probe.predict(X_test)

        accs_k.append(accuracy_score(y_k[test_idx], y_pred))

    acc_by_hop[k] = accs_k
    peak = int(np.nanargmax(accs_k)) + 1
    print(f"  k={k}  L1={accs_k[0]:.3f}  peak=L{peak}({max(accs_k):.3f})"
          f"  L32={accs_k[31]:.3f}")

Running hop analysis for k = [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
  k=1  L1=0.715  peak=L1(0.715)  L32=0.636
  k=2  L1=0.344  peak=L1(0.344)  L32=0.276
  k=3  L1=0.342  peak=L4(0.360)  L32=0.345
  k=4  L1=0.361  peak=L1(0.361)  L32=0.253
  k=5  L1=0.233  peak=L28(0.326)  L32=0.302


In [15]:
# Color ramp: light purple (easy) → dark coral (hard)
colors = ["#AFA9EC", "#7F77DD", "#534AB7", "#D85A30", "#993C1D",
          "#F0997B", "#FAC775", "#1D9E75", "#085041", "#042C53"]

layers = list(range(1, n_layers + 1))

fig = go.Figure()

for i, k in enumerate(hop_values):
    color = colors[i % len(colors)]
    fig.add_trace(go.Scatter(
        x=layers,
        y=acc_by_hop[k],
        name=f"k={k}  (n={int((hops==k).sum())})",
        mode="lines+markers",
        line=dict(color=color, width=2),
        marker=dict(size=4),
        connectgaps=True,
    ))

fig.add_hline(y=0.25, line_dash="dash", line_color="gray", opacity=0.4,
              annotation_text="Chance (0.25)",
              annotation_position="bottom right")

fig.update_layout(
    title="Spatial probe accuracy by layer and hop depth — Pythia-2.8B on StepGame<br>"
          "<sub>Entity-targeted extraction · does early-layer advantage shrink with k?</sub>",
    xaxis_title="Layer",
    yaxis_title="Accuracy",
    yaxis=dict(range=[0, 0.75]),
    legend=dict(title="Hop depth (k)", orientation="v"),
    height=500,
)
fig.show()

In [16]:
# The headline number: does L1 accuracy drop as k increases?

print(f"\n{'k':>4}  {'n':>6}  {'L1 acc':>8}  {'peak layer':>10}  {'L32 acc':>8}")
print("-" * 45)
for k in hop_values:
    accs  = acc_by_hop[k]
    n     = int((hops==k).sum())
    l1    = accs[0]
    peak  = int(np.nanargmax(accs)) + 1
    l32   = accs[31]
    print(f"{k:>4}  {n:>6}  {l1:>8.3f}  {peak:>10}  {l32:>8.3f}")


   k       n    L1 acc  peak layer   L32 acc
---------------------------------------------
   1   14347     0.715           1     0.636
   2    5389     0.344           1     0.276
   3    1927     0.342           4     0.345
   4     788     0.361           1     0.253
   5     213     0.233          28     0.302


In [17]:
print(f"\n{'k':>4}  {'n':>6}  {'L1 acc':>8}  {'peak L':>8}  {'peak acc':>9}  {'L32 acc':>8}  {'above chance?':>14}")
print("-" * 65)
for k in hop_values:
    accs  = acc_by_hop[k]
    n     = int((hops==k).sum())
    l1    = accs[0]
    peak  = int(np.nanargmax(accs)) + 1
    pkval = float(np.nanmax(accs))
    l32   = accs[31]
    above = "YES" if pkval > 0.30 else "marginal"
    print(f"{k:>4}  {n:>6}  {l1:>8.3f}  {peak:>8}  {pkval:>9.3f}  {l32:>8.3f}  {above:>14}")

print(f"\nChance baseline: 0.250")
print(f"\nKey finding: k=1 L1 accuracy ({acc_by_hop[1][0]:.3f}) vs k=2 L1 accuracy "
      f"({acc_by_hop[2][0]:.3f}) — drop of "
      f"{acc_by_hop[1][0] - acc_by_hop[2][0]:.3f} from one additional hop")


   k       n    L1 acc    peak L   peak acc   L32 acc   above chance?
-----------------------------------------------------------------
   1   14347     0.715         1      0.715     0.636             YES
   2    5389     0.344         1      0.344     0.276             YES
   3    1927     0.342         4      0.360     0.345             YES
   4     788     0.361         1      0.361     0.253             YES
   5     213     0.233        28      0.326     0.302             YES

Chance baseline: 0.250

Key finding: k=1 L1 accuracy (0.715) vs k=2 L1 accuracy (0.344) — drop of 0.371 from one additional hop


In [18]:
l1_by_hop  = [acc_by_hop[k][0] for k in hop_values]
l32_by_hop = [acc_by_hop[k][31] for k in hop_values]

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=hop_values, y=l1_by_hop,
    name="Layer 1 accuracy",
    mode="lines+markers",
    line=dict(color="#534AB7", width=2),
    marker=dict(size=8),
))
fig.add_trace(go.Scatter(
    x=hop_values, y=l32_by_hop,
    name="Layer 32 accuracy",
    mode="lines+markers",
    line=dict(color="#D85A30", width=2, dash="dot"),
    marker=dict(size=8),
))
fig.add_hline(y=0.25, line_dash="dash", line_color="gray", opacity=0.5,
              annotation_text="Chance (0.25)",
              annotation_position="bottom right")

fig.update_layout(
    title="Early-layer spatial encoding degrades sharply with hop depth<br>"
          "<sub>Pythia-2.8B · StepGame · entity-targeted extraction</sub>",
    xaxis=dict(title="Hop depth (k)", tickvals=hop_values),
    yaxis=dict(title="Probe accuracy", range=[0, 0.80]),
    legend=dict(orientation="h", yanchor="bottom", y=1.02),
    height=420,
)
fig.show()